# STT 06 — Mesure de vitesse sur le grand dataset (300 steps)

**Pourquoi :** on a choisi un entraînement long (**option 3 : 20 époques**, environ 382 000 steps). Le scheduler cosinus doit connaître le nombre total de steps **dès le départ**. Or l'estimation « environ 1 h par époque » est une **extrapolation** depuis le petit dataset. Avant d'engager environ 20 h de quota, on mesure le **vrai** temps par step.

**Ce qu'on mesure, sans changer aucun réglage** (modèle S, batch de 32, AdamW, lr maximum 1e‑3, warmup de 5 % puis cosinus, clipping à 1, SpecAugment, `num_workers=4`) :

| Mesure | Décision qui en dépend |
|---|---|
| Temps par step (moyenne sur les steps 21 à 300) | Durée d'une époque, et de 20 époques |
| Part du temps passée à **attendre les données** vs à **calculer** sur le GPU | Le CPU (lecture des mp3 + log-Mel) est-il un goulot ? |
| Mémoire GPU maximale | Marge pour un batch ou un modèle plus gros |
| Temps d'une évaluation sur **tout le dev officiel** (16 207 clips) | Peut-on évaluer à chaque époque, ou plus souvent ? |

**Ce n'est pas un entraînement :** 300 steps, c'est 1,6 % de la première époque, en plein warmup. Le modèle n'apprend presque rien et aucun poids n'est sauvegardé.

**Avant de lancer (réglages Kaggle, panneau de droite) :**
1. **Accelerator → GPU T4 x2** (notre code n'utilise qu'un des deux GPU, comme pour les entraînements précédents).
2. **Internet → On.**
3. **Add-ons → Secrets** : coche le Secret de la clé MDC.
4. **Save Version → Save & Run All.** Durée totale estimée : **environ 30 à 40 min** de quota GPU (dont environ 22 min de téléchargement et de décompression).

**Règles de données (conditions MDC) :** les données vont dans `/tmp`, jamais sauvegardé avec le notebook. Aucune phrase n'est affichée. Le notebook reste **privé**.

**À renvoyer :** la sortie de la dernière cellule (le fichier `mesure_vitesse.json` est aussi dans *Output*).

## Cellule 1 — Notre code, et la machine

On clone notre branche, puis on affiche le GPU et le nombre de cœurs CPU (ce sont eux qui préparent les batchs).

In [ ]:
!git clone -b stt-transformer-junior --depth 1 https://github.com/mohamed-943/projet-J-ONE.git
%cd projet-J-ONE/recherche/stt_transformer
!pip install -q datacollective soundfile
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
print("Cœurs CPU :", os.cpu_count())

## Cellule 2 — Téléchargement et décompression

Exactement comme dans l'EDA (`stt_05`). Ce temps (environ 22 min) se répètera **au début de chaque session d'entraînement** : on le mesure à nouveau.

In [ ]:
import os, glob, time, json, subprocess
from kaggle_secrets import UserSecretsClient

# La clé est lue dans le Secret Kaggle : elle n'apparaît jamais dans le code.
# Kaggle cherche le nom EXACT du Secret : on accepte les deux écritures.
secrets = UserSecretsClient()
try:
    os.environ["MDC_API_KEY"] = secrets.get_secret("MDC_API_KEY")
except Exception:
    os.environ["MDC_API_KEY"] = secrets.get_secret("mdc_api_key")
from datacollective import download_dataset

RESULTS = {}                                   # tous les chiffres de la mesure, sauvegardés à la fin
BIG_ID = "cmu5kub4600pkmh078n7n5ogl"           # Scripted Speech 27.0 FR
BIG_DIR = "/tmp/big"                           # /tmp : jamais sauvegardé avec le notebook (conditions MDC)

start = time.time()
download_dataset(BIG_ID, download_directory=BIG_DIR)
archive = glob.glob(f"{BIG_DIR}/*.tar.gz")[0]
RESULTS["telechargement_min"] = round((time.time() - start) / 60, 1)

start = time.time()
subprocess.run(["tar", "-xzf", archive, "-C", BIG_DIR], check=True)
os.remove(archive)                             # libère les 30 Go de l'archive
RESULTS["extraction_min"] = round((time.time() - start) / 60, 1)

BASE = os.path.dirname(glob.glob(f"{BIG_DIR}/**/train.tsv", recursive=True)[0])
print(f"Téléchargement : {RESULTS['telechargement_min']} min | décompression : {RESULTS['extraction_min']} min")

## Cellule 3 — Les données : train et dev officiels

On réutilise **notre** code (`stt/dataset.py`) : `load_split` lit le `.tsv`, normalise le texte et retire les phrases hors vocabulaire. L'EDA a vérifié que sa lecture des `.tsv` est correcte sur le grand dataset.

- **Train :** SpecAugment activé, mélangé (`shuffle=True`), comme dans l'essai 1.
- **Dev officiel :** sans augmentation, dans l'ordre.

Les 113 clips incompatibles avec le CTC ne sont **pas** retirés ici : sur 300 batchs, leur effet sur le temps est nul (et `zero_infinity=True` neutralise leur loss). Le filtre sera ajouté dans le notebook d'entraînement.

In [ ]:
import math
import torch
from torch.utils.data import DataLoader
from stt.dataset import load_split, CommonVoiceDataset, collate_batch

BATCH_SIZE = 32
train_table = load_split([BASE], "train")
dev_table = load_split([BASE], "dev")

train_loader = DataLoader(CommonVoiceDataset(train_table, augment=True), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_batch, num_workers=4)
dev_loader = DataLoader(CommonVoiceDataset(dev_table, augment=False), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate_batch, num_workers=4)

RESULTS["train_clips"] = len(train_table)
RESULTS["dev_clips"] = len(dev_table)
RESULTS["steps_par_epoque"] = math.ceil(len(train_table) / BATCH_SIZE)
print(f"Train : {len(train_table):,} clips -> {RESULTS['steps_par_epoque']:,} steps par époque")
print(f"Dev   : {len(dev_table):,} clips -> {len(dev_loader):,} batchs")

## Cellule 4 — Modèle, loss, optimizer, scheduler

**Les mêmes réglages que l'essai 1.** Seul le nombre total de steps change : il correspond à l'option 3 (20 époques), avec un warmup de 5 %. Pendant ces 300 steps, on est donc au tout début du warmup : le learning rate reste très petit. Ça ne change rien au **temps** d'un step.

In [ ]:
from torch.optim.lr_scheduler import LambdaLR
from stt.model import SpeechTransformer
from stt.training import warmup_cosine

device = "cuda"
torch.manual_seed(0)
model = SpeechTransformer().to(device)                       # taille S : 3 598 322 paramètres
ctc_loss = torch.nn.CTCLoss(blank=0, zero_infinity=True)     # blank = indice 0 du vocabulaire
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)   # weight decay 0,01 par défaut

TOTAL_STEPS = 20 * RESULTS["steps_par_epoque"]               # option 3 : 20 époques
WARMUP_STEPS = int(0.05 * TOTAL_STEPS)                       # 5 %, comme avant
scheduler = LambdaLR(optimizer, lambda step: warmup_cosine(step, WARMUP_STEPS, TOTAL_STEPS))
print(f"Paramètres : {sum(p.numel() for p in model.parameters()):,}")
print(f"Steps prévus : {TOTAL_STEPS:,} dont {WARMUP_STEPS:,} de warmup")

## Cellule 5 — 300 steps chronométrés

Chaque step est découpé en deux temps :
- **attente des données** : le temps pour obtenir le batch suivant du DataLoader et l'envoyer sur le GPU. S'il est grand, ce sont les 4 workers CPU (lecture des mp3 + log-Mel) qui ralentissent tout ;
- **calcul** : forward, loss, backward, clipping, optimizer et scheduler.

**Piège :** le GPU travaille en **asynchrone** : Python continue avant que le calcul soit fini. Sans `torch.cuda.synchronize()`, le chronomètre mesurerait seulement le temps d'*envoyer* les ordres au GPU.

Les **20 premiers steps** ne sont pas comptés : les workers démarrent et le GPU s'initialise, ce qui fausserait la moyenne.

In [ ]:
N_STEPS, SKIP = 300, 20
data_times, compute_times, losses = [], [], []
torch.cuda.reset_peak_memory_stats()
model.train()
loader_iter = iter(train_loader)
for step in range(N_STEPS):
    t0 = time.time()
    batch = next(loader_iter)                                  # attend que les workers aient préparé un batch
    features, feature_lengths, targets, target_lengths = [t.to(device) for t in batch]
    torch.cuda.synchronize()
    t1 = time.time()

    log_probs, out_lengths = model(features, feature_lengths)  # forward
    loss = ctc_loss(log_probs.transpose(0, 1), targets, out_lengths, target_lengths)  # (T, B, C) pour le CTC
    optimizer.zero_grad()
    loss.backward()                                            # backpropagation
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)    # gradient clipping
    optimizer.step()
    scheduler.step()
    torch.cuda.synchronize()                                   # on attend la vraie fin du calcul
    t2 = time.time()

    losses.append(loss.item())
    if step >= SKIP:
        data_times.append(t1 - t0)
        compute_times.append(t2 - t1)
    if (step + 1) % 50 == 0:
        print(f"step {step + 1:3d} | loss {loss.item():.2f} | données {t1 - t0:.3f} s | calcul {t2 - t1:.3f} s")

RESULTS["s_par_step"] = round((sum(data_times) + sum(compute_times)) / len(data_times), 4)
RESULTS["s_attente_donnees_par_step"] = round(sum(data_times) / len(data_times), 4)
RESULTS["s_calcul_par_step"] = round(sum(compute_times) / len(compute_times), 4)
RESULTS["memoire_gpu_max_go"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
RESULTS["loss_debut_fin"] = [round(losses[0], 2), round(losses[-1], 2)]
print(f"\nMoyenne : {RESULTS['s_par_step']} s par step "
      f"(données {RESULTS['s_attente_donnees_par_step']} s + calcul {RESULTS['s_calcul_par_step']} s)")
print(f"Mémoire GPU maximale : {RESULTS['memoire_gpu_max_go']} Go")

## Cellule 6 — Une évaluation sur tout le dev officiel

On chronomètre `evaluate` (dans `stt/training.py`) sur les 16 207 clips : forward seulement, décodage glouton, WER et CER. Le WER obtenu n'a **aucun sens** (le modèle n'a fait que 300 steps) : seul le **temps** compte. Les exemples de phrases renvoyés par `evaluate` ne sont **pas** affichés.

In [ ]:
from stt.training import evaluate

start = time.time()
dev_metrics = evaluate(model, dev_loader, ctc_loss, device)   # aucune phrase affichée
RESULTS["eval_dev_min"] = round((time.time() - start) / 60, 1)
print(f"Évaluation du dev officiel : {RESULTS['eval_dev_min']} min "
      f"(WER {dev_metrics['wer']:.1%}, sans signification après 300 steps)")

## Cellule 7 — Extrapolation et sauvegarde des chiffres

À partir des mesures, on calcule :
- la durée d'**une époque** (entraînement + une évaluation du dev) ;
- la durée des **20 époques** de l'option 3 ;
- le nombre de **sessions** de Kaggle nécessaires (12 h maximum par session, dont le téléchargement et la décompression à chaque fois).

Tout est enregistré dans `/kaggle/working/mesure_vitesse.json` (aucune phrase). **C'est cette sortie qu'il faut me renvoyer.**

In [ ]:
epoch_train_h = RESULTS["steps_par_epoque"] * RESULTS["s_par_step"] / 3600
epoch_h = epoch_train_h + RESULTS["eval_dev_min"] / 60
setup_h = (RESULTS["telechargement_min"] + RESULTS["extraction_min"]) / 60
RESULTS["epoque_entrainement_h"] = round(epoch_train_h, 2)
RESULTS["epoque_avec_eval_h"] = round(epoch_h, 2)
RESULTS["option3_20_epoques_h"] = round(20 * epoch_h, 1)
RESULTS["epoques_par_session_12h"] = round((12 - setup_h) / epoch_h, 1)
RESULTS["sessions_option3"] = math.ceil(20 / max(1, math.floor((12 - setup_h) / epoch_h)))
RESULTS["quota_option3_h"] = round(20 * epoch_h + RESULTS["sessions_option3"] * setup_h, 1)

with open("/kaggle/working/mesure_vitesse.json", "w") as f:
    json.dump(RESULTS, f, indent=1, ensure_ascii=False)
print(json.dumps(RESULTS, indent=1, ensure_ascii=False))